# Downloading weather data
## Important: non-commercial use only since the open-meteo API used is only for non-commercial use

In [16]:
import openmeteo_requests

import requests_cache
from retry_requests import retry

We follow the API request as explained on https://open-meteo.com . Importantly, this request structure is only allowed to be used for non-commercial use

In [17]:
# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
latitudes = list(range(48,55,3))
unix_timestamps = []
hourly_shortwave_radiation = []
hourly_wind_speed_100m =[]

for latitute in latitudes:
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
    	"latitude": latitute,
    	"longitude": 10,   # This is a longitude centrally in Germany. For latitude we use multiple datapoints, but only one longitude so far
    	"start_date": "2022-01-01",
    	"end_date": "2026-08-17",
    	"hourly": ["shortwave_radiation", "wind_speed_100m"],
    }
    responses = openmeteo.weather_api(url, params = params)
    # Process first location. Add a for-loop for multiple locations or weather models
    response = responses[0]
    print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
    print(f"Elevation: {response.Elevation()} m asl")
    print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")
    # Process hourly data. The order of variables needs to be the same as requested.
    hourly = response.Hourly()
    hourly_shortwave_radiation.append( hourly.Variables(0).ValuesAsNumpy() )
    hourly_wind_speed_100m.append( hourly.Variables(1).ValuesAsNumpy() )
    unix_timestamps.append( list(range(
        hourly.Time(), 
        hourly.TimeEnd(), 
        hourly.Interval()
    )))
print(latitudes)
print(unix_timestamps[0][120])

Coordinates: 47.97890853881836°N 10.01661205291748°E
Elevation: 656.0 m asl
Timezone difference to GMT+0: 0s
Coordinates: 51.00175476074219°N 9.982110977172852°E
Elevation: 369.0 m asl
Timezone difference to GMT+0: 0s
Coordinates: 54.02460479736328°N 9.94186019897461°E
Elevation: 24.0 m asl
Timezone difference to GMT+0: 0s
[48, 51, 54]
1641427200


We will now save the downloaded weather data as a .csv

In [18]:
for i in hourly_wind_speed_100m:
    print(i)

[30.760519 28.296091 28.555965 ... 33.30778  31.694824 30.36083 ]
[25.744402 23.688984 25.516865 ... 16.055355 14.028457 12.580699]
[33.933697 32.777798 31.92654  ... 21.026125 18.162169 17.886242]


In [19]:
import csv, sys

outfile = open("weather_data.csv", "w", encoding="utf8")
writer = csv.writer(outfile)

for i,timestamp in enumerate(unix_timestamps[0]):
    data_list = [timestamp*1000] #electricity price data timestamps are also in milliseconds, so we convert the weather data timestamps into milliseconds too
    for j in range(len(latitudes)): # As many data points as the list of latitudes
        data_list.append(hourly_shortwave_radiation[j][i])
    for j in range(len(latitudes)): # Same again for wind speeds
        data_list.append(hourly_wind_speed_100m[j][i])
    writer.writerow(data_list)
outfile.close()

We now downloaded the weather data into a .csv file. In the next notebooks we will combine this data with the electricity price data and use all together for training a price prediction model